# DefectosCafeVerde — FSRC1 frozen-D0 seed 42

Trains only the 1,281-parameter frequency-guided reliability calibrator. The completed D0 detector is frozen; boxes/classes cannot change, test is never extracted, and the train-candidate cache plus calibrator checkpoint are resumable from shared Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import hashlib, importlib, json, os, shutil, subprocess, sys, tarfile, time
WORK=Path('/content'); REPO=WORK/'coffee-bean-detection'
BRANCH='codex/defectoscafeverde-fsrc'
ARCHIVE_NAME='defectoscafeverde-grouped-physical-v1.tar'
EXPECTED_BYTES=686474752
EXPECTED_SHA='53fb2233f1f0d1c77cb24eca2d720f86e0a16835b8a69f4e8f3176fae1aacef2'
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(8*1024*1024),b''): h.update(block)
    return h.hexdigest()
drive_roots=[p for p in [Path('/content/drive/MyDrive'),*Path('/content/drive/.shortcut-targets-by-id').glob('*')] if p.exists()]
archives=sorted({p for root in drive_roots for p in root.rglob(ARCHIVE_NAME) if p.is_file()})
valid=[p for p in archives if p.stat().st_size==EXPECTED_BYTES and sha256(p)==EXPECTED_SHA]
ARCHIVE_ANCHOR=valid[0] if valid else None
if valid:
    ARCHIVE=valid[0]
else:
    complete=[]
    for scheme,count in (('part16',41),('chunk',11),('part',2)):
        by_parent={}
        for root in drive_roots:
            for path in root.rglob(f'{ARCHIVE_NAME}.{scheme}-*'):
                if path.is_file(): by_parent.setdefault(path.parent,{})[path.name]=path
        expected=[f'{ARCHIVE_NAME}.{scheme}-{index:03d}' for index in range(count)]
        for parent,rows in by_parent.items():
            if sorted(rows)==expected and sum(rows[name].stat().st_size for name in expected)==EXPECTED_BYTES: complete.append([rows[name] for name in expected])
    if not complete: raise FileNotFoundError(f'Tidak menemukan {ARCHIVE_NAME} atau set chunk lengkap di shared Drive')
    parts=complete[0]; ARCHIVE_ANCHOR=parts[0]; ARCHIVE=WORK/ARCHIVE_NAME
    with ARCHIVE.open('wb') as target:
        for part in parts:
            with part.open('rb') as source: shutil.copyfileobj(source,target,length=8*1024*1024)
    if ARCHIVE.stat().st_size!=EXPECTED_BYTES or sha256(ARCHIVE)!=EXPECTED_SHA: raise RuntimeError('Archive gabungan tidak valid')
required=Path('experiments/defectoscafeverde-cwcf-direct-v1/val_reports/D0DIRECT_seed42_result.json')
options=[Path('/content/drive/MyDrive/Coffee_Bean_Detection'),Path('/content/drive/.shortcut-targets-by-id/19430gSQnXwxodRUvcPZmXLwpy9Z_h_2c/Coffee_Bean_Detection')]
if ARCHIVE_ANCHOR is not None: options += [p for p in ARCHIVE_ANCHOR.parents if p.name=='Coffee_Bean_Detection']
options=list(dict.fromkeys(p.resolve() for p in options if p.is_dir()))
matching=[p for p in options if (p/required).is_file()]
if not matching: raise FileNotFoundError(f'Folder proyek dengan D0 result tidak ditemukan; kandidat={options}')
PROJECT=matching[0]
D0_RESULT=PROJECT/required
D0_CHECKPOINT=PROJECT/'experiments/defectoscafeverde-cwcf-direct-v1/D0DIRECT/D0DIRECT_seed42/weights/best.pt'
if not D0_CHECKPOINT.is_file(): raise FileNotFoundError(D0_CHECKPOINT)
print('PROJECT:',PROJECT); print('ARCHIVE:',ARCHIVE); print('D0 RESULT:',D0_RESULT); print('D0 CHECKPOINT:',D0_CHECKPOINT)


In [ ]:
os.chdir(WORK)
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True,cwd=WORK)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96'],check=True,cwd=WORK)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True,cwd=WORK)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
import torch
if not torch.cuda.is_available(): raise RuntimeError('Aktifkan GPU T4 sebelum Run All')
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())
print('GPU:',torch.cuda.get_device_name(0))
check=subprocess.run([sys.executable,'-m','pytest','-q','tests/test_defectoscafeverde_fsrc.py'],cwd=REPO,text=True,capture_output=True)
print(check.stdout)
if check.returncode: print(check.stderr); raise RuntimeError('FSRC tests gagal sebelum training')


In [ ]:
DATA=WORK/'defectoscafeverde-development-v1'
if DATA.exists(): shutil.rmtree(DATA)
DATA.mkdir(parents=True)
prefix='grouped-physical-v1/'; allowed={'data.yaml','grouped_audit.json','grouped_summary.json'}
with tarfile.open(ARCHIVE,'r') as bundle:
    for member in bundle.getmembers():
        name=member.name.replace('\\','/')
        if not name.startswith(prefix): continue
        relative=name[len(prefix):].strip('/')
        if not relative: continue
        first=relative.split('/',1)[0]
        if not (first in {'train','val'} or relative in allowed): continue
        target=(DATA/relative).resolve()
        if DATA.resolve() not in target.parents and target!=DATA.resolve(): raise RuntimeError('Unsafe archive path')
        if member.isdir(): target.mkdir(parents=True,exist_ok=True); continue
        target.parent.mkdir(parents=True,exist_ok=True); source=bundle.extractfile(member)
        if source is None: raise RuntimeError(f'Gagal membaca {member.name}')
        with source, target.open('wb') as out: shutil.copyfileobj(source,out)
if (DATA/'test').exists(): raise RuntimeError('TEST TEREXPOSE - STOP')
import yaml
data_yaml=yaml.safe_load((DATA/'data.yaml').read_text()); data_yaml.pop('test',None)
data_yaml['path']=str(DATA); data_yaml['train']='train/images'; data_yaml['val']='val/images'
(DATA/'data.yaml').write_text(yaml.safe_dump(data_yaml,sort_keys=False),encoding='utf-8')
AUDIT=DATA/'grouped_audit.json'
print('TRAIN:',len(list((DATA/'train/images').glob('*'))),'VAL:',len(list((DATA/'val/images').glob('*'))),'TEST:',(DATA/'test').exists())


In [ ]:
from coffee_detector.experiments.run_defectoscafeverde_fsrc import run_fsrc_static_audit
OUT=PROJECT/'experiments/defectoscafeverde-fsrc1-v1'; OUT.mkdir(parents=True,exist_ok=True)
STATIC=OUT/'static_audit.json'
static=run_fsrc_static_audit(STATIC)
print('PARAMETERS:',static['added_parameters']); print('GATES:',static['gates']); print('DECISION:',static['decision'])
assert static['decision']=='PASS','STOP: static audit gagal; training calibrator tidak dijalankan.'
print('PASS: detector tetap frozen; hanya calibrator boleh dilatih.')


In [ ]:
LOG=OUT/'FSRC1_seed42_run.log'
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_defectoscafeverde_fsrc','--data-root',str(DATA),'--grouped-audit',str(AUDIT),'--d0-result',str(D0_RESULT),'--d0-checkpoint',str(D0_CHECKPOINT),'--output-root',str(OUT),'--device','0','--authorize-training']
print('START/RESUME FSRC1 | log=',LOG,flush=True)
with LOG.open('a',encoding='utf-8') as stream:
    process=subprocess.Popen(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT)
shown=None
while process.poll() is None:
    lines=LOG.read_text(errors='replace').splitlines() if LOG.is_file() else []
    status=[line for line in lines if line.startswith(('FSRC CACHE','FSRC CALIBRATOR','FSRC VALIDATION','REUSE FSRC'))]
    if status and status[-1]!=shown: print(status[-1],flush=True); shown=status[-1]
    time.sleep(60)
print('\n'.join(LOG.read_text(errors='replace').splitlines()[-120:]))
if process.returncode: raise RuntimeError(f'FSRC1 gagal: {process.returncode}; log={LOG}')
RESULT=OUT/'FSRC1_seed42_result.json'
if not RESULT.is_file(): raise FileNotFoundError(RESULT)


In [ ]:
from IPython.display import display
import pandas as pd
result=json.loads(RESULT.read_text())
rows=[{'model':model,**values} for model,values in result['values'].items()]
display(pd.DataFrame(rows).style.format({key:'{:.2%}' for key in ('macro_map50_95','bottom3_class_map50_95','worst_class_map50_95')}))
class_rows=[{'class_name':name,'delta':delta} for name,delta in result['class_deltas'].items()]
display(pd.DataFrame(class_rows).sort_values('delta',ascending=False).style.format({'delta':'{:+.2%}'}))
print('DELTAS:',result['comparison']['deltas'])
print('CRITERIA:',result['comparison']['criteria'])
print('GATES:',result['gates'])
print('DECISION:',result['decision'],'| NEXT:',result['next'])
print('TRAIN CANDIDATES:',result['train_candidates'],'| POSITIVE RATE:',result['train_positive_rate'])
print('DETECTOR TRAINING:',result['detector_training_executed'],'| CALIBRATOR TRAINING HISTORICAL:',result['calibrator_training_executed'],'| TRAINING THIS CALL:',result.get('training_executed_this_call'),'| TEST:',result['test_images_accessed'])
print('SUMMARY:',RESULT)
print('Kirim dua tabel, DELTAS, CRITERIA, GATES, DECISION, dan jumlah kandidat. Jangan membuka test.')
